In [1]:
# CELL 1: MASTER CONFIGURATION FOR TRANSFORMER
# This cell defines the specific hyperparameters required for a Transformer model,
# such as the number of attention heads and feed-forward dimensions.

import torch
import torch.nn as nn
import torch.optim as optim
import json
import re
import os
import time
import math
import random
from torch.utils.data import Dataset, DataLoader
from torch.nn.utils.rnn import pad_sequence
from tqdm import tqdm

# --- MASTER SETTINGS ---
CONFIG = {
    "n_epochs": 1,
    "batch_size": 32,
    "emb_dim": 256,         # Embedding size (d_model)
    "n_heads": 8,           # Number of Multi-head Attention heads
    "n_layers": 3,          # Number of Encoder and Decoder layers
    "ff_dim": 512,          # Hidden dimension in Feed-forward network
    "dropout": 0.1,
    "learning_rate": 0.0001,
    "model_path": "atis_transformer.pth",
    "device": torch.device("cuda" if torch.cuda.is_available() else "cpu"),
    "max_sql_len": 120
}

In [2]:
# CELL 2: DATA PREPROCESSING AND SPLIT MANAGEMENT
# This remains similar to the LSTM version, handling tokenization and filtering 
# the ATIS dataset by "train" or "test" splits.

class Vocab:
    def __init__(self, name):
        self.name = name
        self.word2index = {"<PAD>": 0, "<SOS>": 1, "<EOS>": 2, "<UNK>": 3}
        self.index2word = {0: "<PAD>", 1: "<SOS>", 2: "<EOS>", 3: "<UNK>"}
        self.n_words = 4

    def add_sentence(self, sentence):
        for word in self.tokenize(sentence):
            if word not in self.word2index:
                self.word2index[word] = self.n_words
                self.index2word[self.n_words] = word
                self.n_words += 1

    def tokenize(self, text):
        text = text.lower().strip()
        text = re.sub(r"([,.?!\";])", r" \1 ", text)
        return text.split()

def load_split_data(file_path, split_name="train"):
    with open(file_path, 'r') as f:
        data = json.load(f)
    pairs = []
    for item in data:
        sql_query = item['sql'][0] 
        for s in item['sentences']:
            if s['question-split'] == split_name:
                pairs.append((s['text'], sql_query))
    return pairs

class ATISDataset(Dataset):
    def __init__(self, pairs, input_vocab, output_vocab):
        self.pairs = pairs
        self.input_vocab = input_vocab
        self.output_vocab = output_vocab

    def __len__(self): return len(self.pairs)

    def sentence_to_indices(self, vocab, sentence):
        tokens = vocab.tokenize(sentence)
        indices = [vocab.word2index.get(t, vocab.word2index["<UNK>"]) for t in tokens]
        indices.insert(0, vocab.word2index["<SOS>"])
        indices.append(vocab.word2index["<EOS>"])
        return torch.tensor(indices, dtype=torch.long)

    def __getitem__(self, idx):
        src_raw, trg_raw = self.pairs[idx]
        return self.sentence_to_indices(self.input_vocab, src_raw), \
               self.sentence_to_indices(self.output_vocab, trg_raw)

def collate_fn(batch):
    src_batch, trg_batch = zip(*batch)
    return pad_sequence(src_batch, batch_first=True, padding_value=0), \
           pad_sequence(trg_batch, batch_first=True, padding_value=0)

In [3]:
# CELL 3: TRANSFORMER ARCHITECTURE COMPONENTS
# Defines the Positional Encoding (since Transformers have no recurrence) 
# and the Seq2Seq Transformer model using PyTorch's nn.Transformer module.

class PositionalEncoding(nn.Module):
    def __init__(self, d_model, dropout=0.1, max_len=5000):
        super().__init__()
        self.dropout = nn.Dropout(p=dropout)
        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len, dtype=torch.float).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        self.register_buffer('pe', pe.unsqueeze(0))

    def forward(self, x):
        x = x + self.pe[:, :x.size(1)]
        return self.dropout(x)

class ATISTransformer(nn.Module):
    def __init__(self, src_vocab_size, trg_vocab_size, d_model, nhead, num_layers, dim_feedforward, dropout):
        super().__init__()
        self.d_model = d_model
        self.src_embedding = nn.Embedding(src_vocab_size, d_model)
        self.trg_embedding = nn.Embedding(trg_vocab_size, d_model)
        self.pos_encoder = PositionalEncoding(d_model, dropout)
        
        self.transformer = nn.Transformer(
            d_model=d_model, nhead=nhead, num_encoder_layers=num_layers,
            num_decoder_layers=num_layers, dim_feedforward=dim_feedforward,
            dropout=dropout, batch_first=True
        )
        self.fc_out = nn.Linear(d_model, trg_vocab_size)

    def forward(self, src, trg, src_mask=None, trg_mask=None, src_padding_mask=None, trg_padding_mask=None, memory_key_padding_mask=None):
        src_emb = self.pos_encoder(self.src_embedding(src) * math.sqrt(self.d_model))
        trg_emb = self.pos_encoder(self.trg_embedding(trg) * math.sqrt(self.d_model))
        
        output = self.transformer(
            src_emb, trg_emb, tgt_mask=trg_mask, src_key_padding_mask=src_padding_mask,
            tgt_key_padding_mask=trg_padding_mask, memory_key_padding_mask=memory_key_padding_mask
        )
        return self.fc_out(output)

In [5]:
# CELL 4: TRANSFORMER MASKING AND TRAINING UTILITIES
# Transformers require masks to prevent the decoder from looking at future tokens
# and to ensure padding is ignored during the attention mechanism.

def generate_square_subsequent_mask(sz, device):
    mask = (torch.triu(torch.ones((sz, sz), device=device)) == 1).transpose(0, 1)
    mask = mask.float().masked_fill(mask == 0, float('-inf')).masked_fill(mask == 1, float(0.0))
    return mask

def train_epoch(loader, model, optimizer, criterion, config, epoch_idx):
    model.train()
    epoch_loss = 0
    pbar = tqdm(loader, desc=f"Epoch {epoch_idx+1}/{config['n_epochs']} [Train]")
    
    for src, trg in pbar:
        src, trg = src.to(config['device']), trg.to(config['device'])
        
        # In Transformer, we shift target to create input and labels
        trg_input = trg[:, :-1]
        trg_labels = trg[:, 1:]
        
        # Create masks
        trg_mask = generate_square_subsequent_mask(trg_input.size(1), config['device'])
        src_padding_mask = (src == 0)
        trg_padding_mask = (trg_input == 0)

        optimizer.zero_grad()
        output = model(src, trg_input, trg_mask=trg_mask, 
                       src_padding_mask=src_padding_mask, trg_padding_mask=trg_padding_mask,
                       memory_key_padding_mask=src_padding_mask)
        
        loss = criterion(output.reshape(-1, output.shape[-1]), trg_labels.reshape(-1))
        loss.backward()
        optimizer.step()
        epoch_loss += loss.item()
        pbar.set_postfix(loss=loss.item())
        
    return epoch_loss / len(loader)

def save_transformer_checkpoint(model, input_vocab, output_vocab, config, loss):
    checkpoint = {
        'model_state_dict': model.state_dict(),
        'input_vocab': input_vocab,
        'output_vocab': output_vocab,
        'config': config,
        'best_loss': loss
    }
    torch.save(checkpoint, config['model_path'])
    print(f"--- Transformer Model Saved to {config['model_path']} ---")

In [6]:
# CELL 4: TRANSFORMER MASKING AND TRAINING UTILITIES
# Transformers require masks to prevent the decoder from looking at future tokens
# and to ensure padding is ignored during the attention mechanism.

def generate_square_subsequent_mask(sz, device):
    mask = (torch.triu(torch.ones((sz, sz), device=device)) == 1).transpose(0, 1)
    mask = mask.float().masked_fill(mask == 0, float('-inf')).masked_fill(mask == 1, float(0.0))
    return mask

def train_epoch(loader, model, optimizer, criterion, config, epoch_idx):
    model.train()
    epoch_loss = 0
    pbar = tqdm(loader, desc=f"Epoch {epoch_idx+1}/{config['n_epochs']} [Train]")
    
    for src, trg in pbar:
        src, trg = src.to(config['device']), trg.to(config['device'])
        
        # In Transformer, we shift target to create input and labels
        trg_input = trg[:, :-1]
        trg_labels = trg[:, 1:]
        
        # Create masks
        trg_mask = generate_square_subsequent_mask(trg_input.size(1), config['device'])
        src_padding_mask = (src == 0)
        trg_padding_mask = (trg_input == 0)

        optimizer.zero_grad()
        output = model(src, trg_input, trg_mask=trg_mask, 
                       src_padding_mask=src_padding_mask, trg_padding_mask=trg_padding_mask,
                       memory_key_padding_mask=src_padding_mask)
        
        loss = criterion(output.reshape(-1, output.shape[-1]), trg_labels.reshape(-1))
        loss.backward()
        optimizer.step()
        epoch_loss += loss.item()
        pbar.set_postfix(loss=loss.item())
        
    return epoch_loss / len(loader)

def save_transformer_checkpoint(model, input_vocab, output_vocab, config, loss):
    checkpoint = {
        'model_state_dict': model.state_dict(),
        'input_vocab': input_vocab,
        'output_vocab': output_vocab,
        'config': config,
        'best_loss': loss
    }
    torch.save(checkpoint, config['model_path'])
    print(f"--- Transformer Model Saved to {config['model_path']} ---")

In [8]:
# CELL 5: TRANSFORMER EXECUTION AND BEST MODEL SAVING
# Initializes the transformer components and performs split-aware training.
# Saves the best-performing iteration based on the 'test' split.

# 1. Prepare Data
train_pairs = load_split_data('/Users/chillipepper/Documents/Atis Project/atis.json', split_name="train")
test_pairs = load_split_data('/Users/chillipepper/Documents/Atis Project/atis.json', split_name="test")

i_vocab, o_vocab = Vocab("Input"), Vocab("SQL")
for src, trg in train_pairs:
    i_vocab.add_sentence(src)
    o_vocab.add_sentence(trg)

train_loader = DataLoader(ATISDataset(train_pairs, i_vocab, o_vocab), 
                          batch_size=CONFIG['batch_size'], shuffle=True, collate_fn=collate_fn)
test_loader = DataLoader(ATISDataset(test_pairs, i_vocab, o_vocab), 
                         batch_size=CONFIG['batch_size'], shuffle=False, collate_fn=collate_fn)

# 2. Init Model
model = ATISTransformer(
    i_vocab.n_words, o_vocab.n_words, CONFIG['emb_dim'], CONFIG['n_heads'],
    CONFIG['n_layers'], CONFIG['ff_dim'], CONFIG['dropout']
).to(CONFIG['device'])

optimizer = optim.Adam(model.parameters(), lr=CONFIG['learning_rate'], betas=(0.9, 0.98), eps=1e-9)
criterion = nn.CrossEntropyLoss(ignore_index=0)

# 3. Training Loop
best_loss = float('inf')
for epoch in range(CONFIG['n_epochs']):
    train_loss = train_epoch(train_loader, model, optimizer, criterion, CONFIG, epoch)
    
    # Simple Validation check (Evaluation)
    model.eval()
    val_loss = 0
    with torch.no_grad():
        for src, trg in test_loader:
            src, trg = src.to(CONFIG['device']), trg.to(CONFIG['device'])
            ti, tl = trg[:, :-1], trg[:, 1:]
            tm = generate_square_subsequent_mask(ti.size(1), CONFIG['device'])
            out = model(src, ti, trg_mask=tm, src_padding_mask=(src==0), trg_padding_mask=(ti==0), memory_key_padding_mask=(src==0))
            val_loss += criterion(out.reshape(-1, out.shape[-1]), tl.reshape(-1)).item()
    
    avg_val_loss = val_loss / len(test_loader)
    print(f"Epoch {epoch+1} | Train Loss: {train_loss:.4f} | Test Loss: {avg_val_loss:.4f}")
    
    if avg_val_loss < best_loss:
        best_loss = avg_val_loss
        save_transformer_checkpoint(model, i_vocab, o_vocab, CONFIG, best_loss)

Epoch 1/1 [Train]:   0%|          | 0/136 [00:00<?, ?it/s]/opt/anaconda3/lib/python3.12/site-packages/torch/nn/functional.py:5962: UserWarning: Support for mismatched key_padding_mask and attn_mask is deprecated. Use same type for both instead.
  warnings.warn(
Epoch 1/1 [Train]: 100%|██████████| 136/136 [03:17<00:00,  1.45s/it, loss=1.33]
/opt/anaconda3/lib/python3.12/site-packages/torch/nn/modules/transformer.py:505: UserWarning: The PyTorch API of nested tensors is in prototype stage and will change in the near future. We recommend specifying layout=torch.jagged when constructing a nested tensor, as this layout receives active development, has better operator coverage, and works with torch.compile. (Triggered internally at /Users/runner/work/pytorch/pytorch/pytorch/aten/src/ATen/NestedTensorImpl.cpp:182.)
  output = torch._nested_tensor_from_mask(


Epoch 1 | Train Loss: 2.4648 | Test Loss: 1.8678
--- Transformer Model Saved to atis_transformer.pth ---


In [14]:
# CELL 6: TRANSFORMER INFERENCE AND DEPLOYMENT
# This cell provides a callable function to load the saved Transformer model 
# and translate English natural language strings into SQL queries.
torch.serialization.add_safe_globals([Vocab])
def predict_sql_transformer(text, model_path="atis_transformer.pth"):
    """
    Loads the saved model and performs greedy decoding to generate SQL.
    """
    # 1. Load the checkpoint
    if not os.path.exists(model_path):
        return "Model file not found. Please train the model first."
        
    checkpoint = torch.load(model_path,weights_only="False", map_location=CONFIG['device'])
    i_vocab = checkpoint['input_vocab']
    o_vocab = checkpoint['output_vocab']
    conf = checkpoint['config']
    
    # 2. Reconstruct the model architecture
    model = ATISTransformer(
        i_vocab.n_words, o_vocab.n_words, conf['emb_dim'], conf['n_heads'],
        conf['n_layers'], conf['ff_dim'], conf['dropout']
    ).to(conf['device'])
    
    model.load_state_dict(checkpoint['model_state_dict'])
    model.eval()
    
    # 3. Preprocess the input string
    tokens = i_vocab.tokenize(text)
    src_indexes = [i_vocab.word2index.get(t, i_vocab.word2index["<UNK>"]) for t in tokens]
    src_indexes = [i_vocab.word2index["<SOS>"]] + src_indexes + [i_vocab.word2index["<EOS>"]]
    src_tensor = torch.LongTensor(src_indexes).unsqueeze(0).to(conf['device'])
    
    # 4. Greedy Decoding Loop
    # We start with the <SOS> token and predict one token at a time
    trg_indexes = [o_vocab.word2index["<SOS>"]]
    
    for i in range(conf['max_sql_len']):
        trg_tensor = torch.LongTensor(trg_indexes).unsqueeze(0).to(conf['device'])
        
        # Create masks for the current step
        # (src_mask is not needed for single-batch inference if no padding is present)
        trg_mask = generate_square_subsequent_mask(trg_tensor.size(1), conf['device'])
        
        with torch.no_grad():
            output = model(src_tensor, trg_tensor, trg_mask=trg_mask)
        
        # Take the last token from the output sequence
        best_guess = output.argmax(2)[:, -1].item()
        
        # Stop if we hit <EOS>
        if best_guess == o_vocab.word2index["<EOS>"]:
            break
            
        trg_indexes.append(best_guess)
    
    # 5. Convert indices back to SQL string
    predicted_sql = [o_vocab.index2word[idx] for idx in trg_indexes[1:]]
    return " ".join(predicted_sql)

# --- EXAMPLE EXECUTION ---
# Below are sample strings you might find in the ATIS dataset
test_queries = [
    "show flights from Boston to Sydney",
    
]

print("\n--- Model Inference Results ---")
for query in test_queries:
    result = predict_sql_transformer(query)
    print(f"\nInput:  {query}")
    print(f"Output: {result}")


--- Model Inference Results ---

Input:  show flights from Boston to Sydney
Output: select distinct flightalias0 . flight_id from airport_service as airport_servicealias0 , airport_service as airport_servicealias1 , city as cityalias0 , city as cityalias1 , flight as cityalias1 , flight as flightalias0 where ( ( ( ( ( ( ( ( ( ( ( ( ( ( ( ( ( ( ( ( ( ( ( ( ( ( ( ( ( ( ( ( ( ( ( ( ( ( ( ( ( ( ( ( ( ( ( ( ( ( ( ( ( ( ( ( ( ( ( ( ( ( ( ( ( ( ( ( ( ( ( ( ( ( ( ( ( ( ( ( ( ( ( ( ( ( ( ( ( (
